# B.2 Modellkontroll och tolkning

M2 omskolas på hela träningsdatan (2021–2024). Diagnostik (överdispersion) och tolkning (rate ratios, affärsmässig relevans).

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf

data_dir = Path("../../../data")
df = pd.read_csv(data_dir / "Entreprenadförsäkring training.csv")

assert (df["Duration"] > 0).all(), "Duration innehåller nollor — hantera innan log"
df["Ar"] = df["Ar"].astype(int)
df["log_duration"] = np.log(df["Duration"])
df["log_omsattning"] = np.log(df["Omsattning"])

print(f"Hela träningsdatan: {df.shape[0]:,} rader (2021–2024)")

Hela träningsdatan: 1,033,386 rader (2021–2024)


## Omskola M2 på hela träningsdatan

M2 bekräftad i B1. Omskolas på 2021–2024 för bästa koefficientskattningar.

In [2]:
m2_full = smf.glm(
    "AntalSkador ~ C(Verksamhet, Treatment(reference='Byggföretag')) "
    "+ C(GeografisktOmrade, Treatment(reference='Landsbyggd')) "
    "+ log_omsattning",
    data=df,
    family=sm.families.Poisson(),
    offset=df["log_duration"],
).fit()

print(m2_full.summary())

                 Generalized Linear Model Regression Results                  
Dep. Variable:            AntalSkador   No. Observations:              1033386
Model:                            GLM   Df Residuals:                  1033375
Model Family:                 Poisson   Df Model:                           10
Link Function:                    Log   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -94353.
Date:                Thu, 23 Apr 2026   Deviance:                   1.4963e+05
Time:                        12:54:35   Pearson chi2:                 1.02e+06
No. Iterations:                     7   Pseudo R-squ. (CS):           0.005575
Covariance Type:            nonrobust                                         
                                                                                    coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------

## 1. Överdispersionskontroll

Poisson antar var(Y) = E(Y). Kontrolleras med Pearson χ²/frihetsgrader — kvot nära 1 innebär att antagandet håller.

In [3]:
pearson_chi2 = m2_full.pearson_chi2
df_resid = m2_full.df_resid
dispersion = pearson_chi2 / df_resid

print(f"Pearson χ²:         {pearson_chi2:,.2f}")
print(f"Frihetsgrader:      {df_resid:,}")
print(f"Dispersionskvot:    {dispersion:.4f}")

if dispersion > 1.5:
    print("\n→ Dispersionskvoten är klart större än 1 — överdispersion föreligger.")
    print("  Standardfel kan vara underskattade. Negativ binomial är ett alternativ.")
elif dispersion > 1.1:
    print("\n→ Måttlig överdispersion. Poisson-modellen är rimlig men standardfelen")
    print("  kan vara något underskattade.")
else:
    print("\n→ Dispersionskvoten ligger nära 1 — ingen allvarlig överdispersion.")

Pearson χ²:         1,018,750.47
Frihetsgrader:      1,033,375
Dispersionskvot:    0.9858

→ Dispersionskvoten ligger nära 1 — ingen allvarlig överdispersion.


Dispersionskvot ≈ 0,99 — ingen överdispersion. Poisson-modellen är adekvat.

## 2. Rate ratios med konfidensintervall

In [4]:
params = m2_full.params
conf = m2_full.conf_int()

rr_table = pd.DataFrame({
    "Koefficient (β)": params.values,
    "Rate ratio": np.exp(params.values),
    "KI nedre (2.5%)": np.exp(conf.iloc[:, 0].values),
    "KI övre (97.5%)": np.exp(conf.iloc[:, 1].values),
}, index=params.index)

# Rensa variabelnamn för läsbarhet
clean_names = (
    rr_table.index
    .str.replace(r"C\(Verksamhet, Treatment\(reference='Byggföretag'\)\)\[T\.", "", regex=True)
    .str.replace(r"C\(GeografisktOmrade, Treatment\(reference='Landsbyggd'\)\)\[T\.", "", regex=True)
    .str.rstrip("]")
)
rr_table.index = clean_names
rr_table.index.name = "Variabel"

display(rr_table.round(4))

,Koefficient (β),Rate ratio,KI nedre (2.5%),KI övre (97.5%)
Variabel,,,,
Intercept,-11.1146,0.0000,0.0000,0.0000
Elektriker,-0.3602,0.6975,0.6593,0.7379
Grävning & Schaktning,-0.1568,0.8549,0.8078,0.9047
Målare,-0.4517,0.6366,0.6005,0.6748
Takarbeten,0.1266,1.1350,1.0671,1.2072
VVS,0.3588,1.4316,1.3724,1.4935
Övriga specialistföretag,-0.0304,0.9701,0.9319,1.0098
Mellanstorstad,0.1851,1.2033,1.1395,1.2708
Småstad,-0.2789,0.7566,0.7110,0.8051


### Verksamhet (relativt Byggföretag)

VVS och Takarbeten har rate ratio > 1 (högre risk). Målare och Elektriker < 1 (lägre risk). Konsistent med deskriptiv analys.

### Geografi (relativt Landsbyggd)

Storstad klart högst, Småstad under Landsbyggd. Geografisk effekt kvarstår efter kontroll för verksamhet och omsättning.

### log(Omsättning) — effekt per fördubbling

Rate ratio per fördubbling = exp(β × log(2)).

In [5]:
beta_oms = m2_full.params["log_omsattning"]
ci_oms = m2_full.conf_int().loc["log_omsattning"]

rr_doubling = np.exp(beta_oms * np.log(2))
rr_doubling_lower = np.exp(ci_oms.iloc[0] * np.log(2))
rr_doubling_upper = np.exp(ci_oms.iloc[1] * np.log(2))

print(f"Koefficient för log(Omsättning):        {beta_oms:.4f}")
print(f"Rate ratio per fördubbling:              {rr_doubling:.4f}")
print(f"95 % KI:                                 [{rr_doubling_lower:.4f}, {rr_doubling_upper:.4f}]")
print(f"\nTolkning: En fördubbling av omsättningen hänger samman med "
      f"{(rr_doubling - 1) * 100:.1f} % högre skadefrekvens.")

Koefficient för log(Omsättning):        0.4416
Rate ratio per fördubbling:              1.3581
95 % KI:                                 [1.3451, 1.3713]

Tolkning: En fördubbling av omsättningen hänger samman med 35.8 % högre skadefrekvens.


## 3. Affärsmässig tolkning

**Verksamhet:** Spridningen mellan högsta (VVS, RR ~1,43) och lägsta (Målare, RR ~0,64) ger en faktor ~2,2 i premiedifferentiering.

**Geografi:** Storstad (RR ~1,46) vs Småstad (RR ~0,76) — faktor ~1,9. Motiverar geografisk differentiering.

**Omsättning:** Fördubbling av omsättning → ~36 % högre skadefrekvens. Premien bör skala mer än linjärt med omsättning. Log-transformen innebär avtagande marginaleffekt — rimligt affärsmässigt.

Resultaten ska tolkas som samband, inte orsakssamband. Skillnader kan också spegla kundmix, rapporteringsbenägenhet eller säkerhetsrutiner som inte finns i datan.

## 4. Sammanfattning

- Överdispersion: kvot ≈ 0,99 → Poisson adekvat
- Alla tre variabelgrupper (verksamhet, geografi, omsättning) är relevanta ratingfaktorer
- Rate ratios kan översättas direkt till premiejusteringar